# Turn census polygons into an H3 crosswalk

A census tract is one polygon row. Most Polars operations are easier when spatial observations share a compact key. This notebook turns WGS84 tract polygons into H3 cells while keeping the transformation inspectable.

By the end, we will have an analysis-ready crosswalk with one tract/cell pair per row, a cell-level ownership table with one row per H3 cell, and a dissolved polygon reconstructed from a cell list. The goal is geometry and data modeling—not a synthetic risk score.

## The route from polygons to ordinary Polars tables

```text
GeoParquet WKB polygons
        │
        ├── polygon_to_cells ──> List(UInt64) coverage per tract
        │                              │
        │                              ├── explode ──> tract/cell crosswalk
        │                              │
        │                              └── cells_to_multi_polygon_wkt
        │                                         │
        └──────────────── visual audit <──────────┘
```

Every important transformation is followed by a count, uniqueness check, round trip, or map. From this repository, run `uv sync` and then `uv run jupyter lab notebooks/geometry.ipynb`. For a standalone installation, install `polars`, `polars-h3`, `folium`, and `matplotlib`.

In [62]:
from pathlib import Path

import polars as pl
from IPython.display import display

import polars_h3 as plh3


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").is_file() and (candidate / "polars_h3").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the polars-h3 checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRACTS_PATH = (
    PROJECT_ROOT / "notebooks" / "data" / "houston-population-tracts-2020.geoparquet"
)

CANDIDATE_RESOLUTIONS = (7, 8, 9)
RESOLUTION = 9

assert TRACTS_PATH.is_file(), f"Missing sample data: {TRACTS_PATH}"

## Meet the source at its native grain

The checked-in file contains Houston-area census tracts. Its GeoParquet metadata describes WGS84 geometry, while Polars exposes the physical geometry column as ordinary WKB `Binary`. That is exactly what `polygon_to_cells` accepts—no GeoPandas object or special Polars geometry dtype is required.

CRS remains the caller's responsibility: standard WKB bytes do not carry enough information for the expression to reproject coordinates.

In [63]:
tracts = pl.read_parquet(TRACTS_PATH)
assert tracts.schema["geometry"] == pl.Binary
assert tracts["geoid"].n_unique() == tracts.height

display(
    tracts.select(
        pl.len().alias("tracts"),
        pl.col("geoid").n_unique().alias("unique_geoids"),
        pl.col("population_estimate").sum().alias("estimated_population"),
        pl.col("land_area_sq_km").sum().alias("land_area_sq_km"),
    )
)
display(
    tracts.select(
        "geoid",
        "tract_name",
        "population_estimate",
        "population_density_per_sq_km",
    ).head(3)
)

tracts,unique_geoids,estimated_population,land_area_sq_km
u32,u32,i64,f64
1421,1421,6277630,10162.031431


geoid,tract_name,population_estimate,population_density_per_sq_km
str,str,i64,f64
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797
"""48039660200""","""Census Tract 6602, Brazoria Co…",6806,375.693731
"""48039660301""","""Census Tract 6603.01, Brazoria…",3365,1061.601267


## Choose a spatial grain by inspecting the result

H3 resolution is a modeling choice, not a formatting option. `polygon_to_cells` uses centroid containment: a cell is selected when its centroid lies inside the polygon. At a coarse resolution, a small tract may therefore receive no cells.

We compare three candidates before choosing one. `feature_cell_rows` is the number of tract/cell pairs we would get after exploding the lists.

In [64]:
def summarize_coverage(resolution: int) -> pl.DataFrame:
    return tracts.select(h3_cells=plh3.polygon_to_cells("geometry", resolution)).select(
        pl.lit(resolution).alias("resolution"),
        pl.col("h3_cells").list.len().sum().alias("feature_cell_rows"),
        pl.col("h3_cells").list.len().median().alias("median_cells_per_tract"),
        (pl.col("h3_cells").list.len() == 0).sum().alias("empty_tracts"),
    )


resolution_check = pl.concat(
    [summarize_coverage(resolution) for resolution in CANDIDATE_RESOLUTIONS]
).sort("resolution")
display(resolution_check)

selected_empty_tracts = resolution_check.filter(
    pl.col("resolution") == RESOLUTION
).item(0, "empty_tracts")
assert selected_empty_tracts == 0

resolution,feature_cell_rows,median_cells_per_tract,empty_tracts
i32,u32,f64,u32
7,1922,0.0,733
8,13503,3.0,88
9,94442,21.0,0


Resolution 9 is the first tested grain that represents every tract. That does not make it universally correct; it makes the reason for this notebook's choice observable.

In [65]:
tracts_covered = tracts.with_columns(
    h3_cells=plh3.polygon_to_cells("geometry", RESOLUTION)
).with_columns(cell_count=pl.col("h3_cells").list.len())

display(
    tracts_covered.select(
        pl.len().alias("tracts"),
        pl.col("cell_count").sum().alias("tract_cell_pairs"),
        pl.col("cell_count").min().alias("min_cells_per_tract"),
        pl.col("cell_count").median().alias("median_cells_per_tract"),
        pl.col("cell_count").max().alias("max_cells_per_tract"),
    )
)

tracts,tract_cell_pairs,min_cells_per_tract,median_cells_per_tract,max_cells_per_tract
u32,u32,u32,f64,u32
1421,94442,1,21.0,5110


## Audit one polygon before scaling out

A list of cell IDs is hard to reason about by inspection. Choose a tract near the median coverage size and overlay its source boundary with the generated cells. Toggle the two layers and zoom along the edges. The small boundary differences are the visible consequence of representing a polygon with whole H3 cells.

In [66]:
median_cell_count = int(tracts_covered["cell_count"].median())
audit_tract = tracts_covered.filter(pl.col("cell_count") == median_cell_count).head(1)

display(
    audit_tract.select(
        "geoid",
        "tract_name",
        "cell_count",
    )
)
display(
    plh3.graphing.plot_polygon_coverage(
        audit_tract,
        geometry_col="geometry",
        cells_col="h3_cells",
    )
)

geoid,tract_name,cell_count
str,str,u32
"""48039660804""","""Census Tract 6608.04, Brazoria…",21


## Close the geometry round trip

`cells_to_multi_polygon_wkt` dissolves a list of cells: shared internal cell edges disappear and the exterior becomes a WKT `MULTIPOLYGON`. Covering that reconstructed polygon again at the same resolution should recover the original cell set.

This round trip validates the cell representation; it does not claim that the H3 outline is identical to the original tract boundary.

In [67]:
audit_roundtrip = (
    audit_tract.with_columns(
        reconstructed_geometry=plh3.cells_to_multi_polygon_wkt("h3_cells")
    )
    .with_columns(
        recovered_cells=plh3.polygon_to_cells("reconstructed_geometry", RESOLUTION)
    )
    .with_columns(
        roundtrip_matches=(
            pl.col("h3_cells").list.sort() == pl.col("recovered_cells").list.sort()
        )
    )
)

assert audit_roundtrip.item(0, "roundtrip_matches")
display(
    audit_roundtrip.select(
        "geoid",
        "cell_count",
        "roundtrip_matches",
        pl.col("reconstructed_geometry").str.slice(0, 90).alias("wkt_preview"),
    )
)

geoid,cell_count,roundtrip_matches,wkt_preview
str,u32,bool,str
"""48039660804""",21,true,"""MULTIPOLYGON(((-95.30464339629…"


## Explode into a tract-to-cell crosswalk

The list column is ideal while each source polygon should remain one row. For joins and group-bys, explode it into one row per tract/cell pair.

Notice that tract attributes repeat after the explode. The repeated population and density values still describe the source tract; they have not become cell-level estimates. Summing them across the exploded rows would overcount the source data.

In [68]:
tract_cell_crosswalk = (
    tracts_covered.select(
        "geoid",
        "tract_name",
        "population_estimate",
        "population_density_per_sq_km",
        "h3_cells",
    )
    .explode("h3_cells")
    .rename({"h3_cells": "h3_cell"})
    .with_columns(h3_cell_hex=plh3.int_to_str("h3_cell"))
)

expected_pairs = int(tracts_covered["cell_count"].sum())
assert tract_cell_crosswalk.height == expected_pairs
assert (
    tract_cell_crosswalk.select("geoid", "h3_cell").unique().height
    == tract_cell_crosswalk.height
)

display(tract_cell_crosswalk.head(8))

geoid,tract_name,population_estimate,population_density_per_sq_km,h3_cell,h3_cell_hex
str,str,i64,f64,u64,str
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196844753977343,"""89446c144d3ffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845250478079,"""89446c1626bffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845250740223,"""89446c1626fffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845480902655,"""89446c17027ffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845495320575,"""89446c17103ffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845495582719,"""89446c17107ffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845495844863,"""89446c1710bffff"""
"""48039660100""","""Census Tract 6601, Brazoria Co…",4938,1168.268797,618196845496107007,"""89446c1710fffff"""


## Make the intended key cardinality explicit

The crosswalk is unique on `(geoid, h3_cell)`, not necessarily on `h3_cell`. Source polygons can overlap, and centroid-based coverage can therefore assign one cell to more than one tract.

When downstream work needs one row per cell, group deliberately and retain the contributing GEOIDs instead of silently dropping a duplicate.

In [69]:
cell_ownership = tract_cell_crosswalk.group_by("h3_cell").agg(
    geoids=pl.col("geoid").unique().sort(),
    tract_count=pl.col("geoid").n_unique(),
)

assert cell_ownership["h3_cell"].n_unique() == cell_ownership.height
display(
    cell_ownership.select(
        pl.len().alias("unique_h3_cells"),
        (pl.col("tract_count") > 1).sum().alias("cells_with_multiple_tracts"),
        pl.col("tract_count").max().alias("max_tracts_per_cell"),
    )
)

unique_h3_cells,cells_with_multiple_tracts,max_tracts_per_cell
u32,u32,u32
94387,55,2


## Re-express a source attribute without inventing precision

For a final visual, carry the tract-level population density onto its H3 cells. This is a change of spatial key, not a new fine-grained estimate: every cell from the same tract receives the same source value. Cells claimed by multiple sample tracts are omitted because choosing how to combine their attributes is a separate analytical decision.

The blocky edges are useful. They remind us which detail came from the H3 representation and which detail was actually present in the census data.

In [70]:
MAP_GEOIDS = (
    "48201310101",
    "48201310102",
    "48201310200",
    "48201312300",
    "48201312501",
    "48201312502",
    "48201410101",
    "48201410102",
    "48201410201",
    "48201410202",
    "48201410501",
    "48201410502",
)

map_tracts = tracts_covered.filter(pl.col("geoid").is_in(MAP_GEOIDS))
assert map_tracts.height == len(MAP_GEOIDS)

density_cells = (
    tract_cell_crosswalk.group_by("h3_cell")
    .agg(
        source_population_density=pl.col("population_density_per_sq_km").first(),
        tract_count=pl.col("geoid").n_unique(),
    )
    .filter(pl.col("tract_count") == 1)
)

map_cell_ids = (
    tract_cell_crosswalk.filter(pl.col("geoid").is_in(MAP_GEOIDS))
    .select("h3_cell")
    .unique()
)
map_cells = density_cells.join(map_cell_ids, on="h3_cell", how="semi")

density_map = plh3.graphing.plot_polygon_coverage(
    map_tracts,
    geometry_col="geometry",
    cells_col="h3_cells",
    cell_fill_opacity=0.0,
    geometry_color="#64748b",
)
density_map = plh3.graphing.plot_hex_fills(
    map_cells,
    hex_id_col="h3_cell",
    metric_col="source_population_density",
    map=density_map,
    map_size="large",
)
display(density_map)

## Optional extension: soften arbitrary tract boundaries

Census tracts are statistical subdivisions with relatively stable boundaries. They are useful, but their variable sizes and boundaries can shape the pattern we see—a version of the **modifiable areal unit problem**. Once every observation has an H3 key, neighborhood operations become ordinary list, join, and aggregation work.

Below, each target cell looks up the cells within two H3 steps. The source cell receives weight 4, first-ring neighbors weight 2, and second-ring neighbors weight 1. Missing neighbors are ignored and the available weights are renormalized, so the result stays on the existing census coverage instead of expanding beyond it.

We smooth population **density**, an intensive measure, with a weighted mean. This is a descriptive surface—not a claim about where people live inside a tract. Smoothing additive totals such as population would require a different, conservation-aware method.

In [71]:
SMOOTHING_RADIUS = 2
smoothing_weights = pl.DataFrame(
    {
        "distance": [0, 1, 2],
        "weight": [4.0, 2.0, 1.0],
    },
    schema={"distance": pl.Int64, "weight": pl.Float64},
)

smoothed_density_cells = (
    density_cells.select(pl.col("h3_cell").alias("target_cell"))
    .with_columns(neighbor_cell=plh3.grid_disk("target_cell", SMOOTHING_RADIUS))
    .explode("neighbor_cell")
    .with_columns(distance=plh3.grid_distance("target_cell", "neighbor_cell"))
    .join(
        density_cells.select(
            pl.col("h3_cell").alias("neighbor_cell"),
            pl.col("source_population_density").alias("neighbor_density"),
        ),
        on="neighbor_cell",
        how="inner",
    )
    .join(smoothing_weights, on="distance", how="inner")
    .group_by("target_cell")
    .agg(
        smoothed_population_density=(
            (pl.col("neighbor_density") * pl.col("weight")).sum()
            / pl.col("weight").sum()
        ),
        contributing_cells=pl.len(),
    )
    .rename({"target_cell": "h3_cell"})
    .join(
        density_cells.select("h3_cell", "source_population_density"),
        on="h3_cell",
        how="left",
    )
)

assert smoothed_density_cells.height == density_cells.height
map_smoothed_cells = smoothed_density_cells.join(map_cell_ids, on="h3_cell", how="semi")

display(
    map_smoothed_cells.select(
        pl.len().alias("mapped_cells"),
        pl.col("contributing_cells").min().alias("min_neighbors_used"),
        pl.col("contributing_cells").median().alias("median_neighbors_used"),
        (pl.col("smoothed_population_density") - pl.col("source_population_density"))
        .abs()
        .mean()
        .alias("mean_absolute_change"),
    )
)

smoothed_map = plh3.graphing.plot_polygon_coverage(
    map_tracts,
    geometry_col="geometry",
    cells_col="h3_cells",
    cell_fill_opacity=0.0,
    geometry_color="#64748b",
)
smoothed_map = plh3.graphing.plot_hex_fills(
    map_smoothed_cells,
    hex_id_col="h3_cell",
    metric_col="smoothed_population_density",
    map=smoothed_map,
    map_size="large",
)
display(smoothed_map)

mapped_cells,min_neighbors_used,median_neighbors_used,mean_absolute_change
u32,u32,f64,f64
105,19,19.0,437.596949


## What the workflow established

- WGS84 WKB polygons went directly from GeoParquet into a Polars expression.
- Resolution selection was checked against empty coverage rather than chosen by habit.
- The source polygon and H3 approximation stayed visually auditable.
- Cell lists became a validated tract/cell crosswalk with explicit key cardinality.
- Dissolving and re-covering a cell set reproduced the same H3 cells.
- Source attributes remained labeled as tract-level values after changing spatial keys.
- A weighted neighborhood mean demonstrated what the shared H3 key enables downstream.

H3 did not preserve the exact tract boundary, make census attributes more spatially precise, choose an overlap rule, or decide the best resolution. Those remain visible modeling decisions.

### Things to try

- Set `RESOLUTION = 8`, rerun, and inspect the empty-tract and boundary checks.
- Choose the smallest or largest tract instead of the median one for the visual audit.
- Join another table to `tract_cell_crosswalk` on `geoid` or to `cell_ownership` on `h3_cell`.
- Dissolve a multi-tract cell set and compare its exterior with the source polygons.
- Change the smoothing radius or weights and inspect how much boundary structure remains.